<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 01. Métricas de Distancia: ¿Qué Tan Parecidos Son Dos Datos?
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 10
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/10%20-%20Clustering/Para%20Dummies/01_Metricas_de_Distancia_y_Estandarizacion_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del módulo 01 de Clustering. El cuaderno principal usó fórmulas como $\sqrt{\sum (p_i-q_i)^2}$ y palabras como "norma $L_2$" — aquí vamos a entender lo mismo, pero con ejemplos de la calle y del barrio.

Al terminar podrás explicar, con tus propias palabras:
1. Por qué un algoritmo de clustering necesita una forma de medir "qué tan parecidos" son dos datos.
2. Qué es la distancia Euclidiana ("línea recta") y la distancia Manhattan ("como un taxi").
3. Qué es la distancia Coseno y por qué le importa la *dirección* y no el *tamaño*.
4. Qué es la distancia de Hamming, para comparar textos o códigos.
5. Por qué casi siempre hay que **estandarizar** las variables antes de calcular distancias.

---
## 1. ¿Cómo le explicas a un computador qué es "parecido"? 🧭

Para nosotros es fácil decir que dos personas "se parecen" en edad e ingresos. Pero un computador no tiene intuición — solo sabe hacer cuentas con números. Así que necesitamos convertir la idea de "parecido" en una **fórmula matemática**: eso es justamente una **métrica de distancia**.

Cuanto **menor** sea la distancia calculada entre dos puntos, más "parecidos" los considera el algoritmo. Cuanto **mayor** la distancia, más diferentes. Todo el clustering se apoya en esta idea — por eso la métrica que elijamos importa tanto: cambiar de métrica puede cambiar por completo qué grupos descubre el algoritmo.

---
## Configuración del entorno de trabajo 🛠️

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import euclidean, cityblock, cosine
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances

df_mall = pd.read_csv('../data/mall_customers.csv')
print("Dataset cargado:", df_mall.shape)

---
## 2. Distancia Euclidiana: la línea recta ("como vuela el pájaro") 🐦

Si dos amigos quedan de verse en un parque abierto y caminan en línea recta el uno hacia el otro, la distancia que recorren es la **distancia Euclidiana**: la línea recta más corta entre dos puntos, la misma que mides con una regla sobre un mapa.

Para dos puntos $p$ y $q$, se calcula así: se restan sus coordenadas, se elevan al cuadrado, se suman, y se saca la raíz cuadrada. Vamos a calcularla entre dos clientes de ejemplo.

In [ ]:
cliente_p = np.array([2, 3])   # ejemplo: (edad reescalada, ingreso reescalado)
cliente_q = np.array([8, 7])

d_manual = np.sqrt(np.sum((cliente_p - cliente_q) ** 2))
d_scipy = euclidean(cliente_p, cliente_q)

print(f"Distancia Euclidiana (a mano):        {d_manual:.4f}")
print(f"Distancia Euclidiana (con scipy):     {d_scipy:.4f}")

### 🤔 ¿Qué acaba de pasar?

- Definimos dos puntos de ejemplo, `cliente_p` y `cliente_q`, cada uno con dos números (dos "características").
- El cálculo a mano (`np.sqrt(np.sum((p - q) ** 2))`) sigue exactamente la fórmula: resta, eleva al cuadrado, suma, saca raíz.
- `scipy.spatial.distance.euclidean` hace lo mismo por nosotros, y confirmamos que da el mismo resultado.
- Esta es la métrica de distancia más intuitiva, y la que usa el algoritmo K-Means que veremos en el siguiente cuaderno.

---
## 3. Distancia Manhattan: moverse como un taxi 🚕

En una ciudad con calles en cuadrícula (como Manhattan, en Nueva York), un taxi no puede atravesar los edificios en línea recta — tiene que moverse por las calles: tantas cuadras al norte, tantas cuadras al este. La **distancia Manhattan** suma esos movimientos "en cuadrícula" en lugar de medir la línea recta.

Es más "tolerante" con valores atípicos que la Euclidiana, porque no eleva las diferencias al cuadrado (un valor muy alejado no se "castiga" tan fuerte).

In [ ]:
d_manhattan_manual = np.sum(np.abs(cliente_p - cliente_q))
d_manhattan_scipy = cityblock(cliente_p, cliente_q)

print(f"Distancia Manhattan (a mano):   {d_manhattan_manual}")
print(f"Distancia Manhattan (scipy):    {d_manhattan_scipy}")
print(f"Recordatorio -> Distancia Euclidiana entre los mismos puntos: {d_manual:.4f}")

### 🤔 ¿Qué acaba de pasar?

- Con los mismos dos puntos de antes, sumamos el valor absoluto de cada diferencia de coordenada — sin elevar al cuadrado ni sacar raíz.
- Nota que el resultado (8) es distinto al de la distancia Euclidiana (~7.2): no hay una única forma "correcta" de medir distancia, cada una captura una noción distinta de "parecido".
- En la práctica, Manhattan suele preferirse cuando los datos tienen valores atípicos (outliers) que no queremos que dominen el cálculo.

---
## 4. Distancia Coseno: ¿misma dirección, aunque distinto tamaño? 📐

Imagina dos flechas dibujadas desde el mismo punto de partida. La distancia coseno no mide qué tan largas son las flechas, sino **qué tan parecido es el ángulo** entre ellas. Dos flechas que apuntan casi al mismo lugar tienen distancia coseno cercana a 0, sin importar si una es mucho más larga que la otra.

Esto es muy útil, por ejemplo, para comparar documentos de texto: un resumen corto y un artículo largo sobre el mismo tema pueden "apuntar en la misma dirección" (hablar de lo mismo, en proporciones parecidas) aunque tengan tamaños (magnitudes) muy distintos.

In [ ]:
documento_a = np.array([4, 0, 1])    # frecuencias de palabras clave, documento largo
documento_b = np.array([2, 0, 0.5])  # mismo tema, documento más corto (mitad de longitud)

d_cos = cosine(documento_a, documento_b)
d_euclid = euclidean(documento_a, documento_b)

print(f"Distancia Coseno entre A y B:     {d_cos:.4f}")
print(f"Distancia Euclidiana entre A y B: {d_euclid:.4f}")

### 🤔 ¿Qué acaba de pasar?

- `documento_b` tiene exactamente las mismas proporciones que `documento_a`, solo que a la mitad de tamaño (como si fuera un resumen del mismo texto).
- La distancia Coseno da un valor muy cercano a 0: para esta métrica, ambos "apuntan casi en la misma dirección", así que los considera prácticamente iguales.
- La distancia Euclidiana, en cambio, es bastante mayor: como sí le importa el tamaño (la magnitud), ve a los dos documentos como bastante distintos.
- Por eso la distancia Coseno es tan usada en textos y sistemas de recomendación: lo que importa ahí es la *proporción*, no el tamaño absoluto.

---
## 5. Distancia de Hamming: contando diferencias letra por letra 🔤

La **distancia de Hamming** sirve para comparar cosas de **igual longitud**, como dos palabras del mismo tamaño o dos códigos binarios: simplemente cuenta en cuántas posiciones son distintas.

Por ejemplo, `"CASA"` y `"CASO"` solo difieren en la última letra → distancia de Hamming = 1.

⚠️ Solo funciona si ambas secuencias tienen la misma longitud — no sirve para comparar `"MARIA"` (5 letras) con `"MARA"` (4 letras) directamente.

In [ ]:
def distancia_hamming(s1: str, s2: str) -> int:
    if len(s1) != len(s2):
        raise ValueError("La distancia de Hamming necesita cadenas de igual longitud.")
    return sum(c1 != c2 for c1, c2 in zip(s1, s2))

print("Hamming('CASA', 'CASO')   =", distancia_hamming('CASA', 'CASO'))
print("Hamming('10101', '01101') =", distancia_hamming('10101', '01101'))

### 🤔 ¿Qué acaba de pasar?

- La función recorre las dos cadenas letra por letra (`zip`) y cuenta cuántas posiciones no coinciden.
- `"CASA"` vs. `"CASO"` difieren solo en la última letra → 1 diferencia.
- `"10101"` vs. `"01101"` difieren en la primera y segunda posición → 2 diferencias.
- Esta métrica es típica para comparar códigos, secuencias de ADN simplificadas, o cualquier dato categórico de longitud fija.

---
## 6. ¿Por qué hay que "emparejar las unidades" antes de comparar? ⚖️

Imagina que quieres comparar qué tan parecidos son dos ingredientes de una receta: uno se mide en **kilogramos** y el otro en **gramos**. Si comparas los números "en crudo" (sin fijarte en la unidad), el que está en gramos parecerá gigantesco frente al que está en kilogramos, aunque en la vida real sean cantidades parecidas.

Algo muy similar pasa con `Age` (edad, números pequeños, como 20-70) y `Annual_Income_k` (ingreso en miles, números mucho más grandes, como 15-140) en nuestro dataset: si calculamos la distancia Euclidiana directamente, el ingreso **dominará** el resultado solo por tener números más grandes — no porque sea realmente "más importante" para decidir qué tan parecidos son dos clientes.

La solución es **estandarizar**: transformar cada columna para que tenga media 0 y una dispersión comparable, usando `StandardScaler`. Así todas las variables "hablan en las mismas unidades" antes de medir distancias.

In [ ]:
columnas = ['Age', 'Annual_Income_k']
X_crudo = df_mall[columnas].values

escalador = StandardScaler()
X_escalado = escalador.fit_transform(X_crudo)

# Distancias entre los primeros 5 clientes, sin escalar vs. escaladas
dist_cruda = euclidean_distances(X_crudo[:5])
dist_escalada = euclidean_distances(X_escalado[:5])

print("Distancias SIN escalar (el ingreso domina por tener números más grandes):")
print(np.round(dist_cruda, 2))
print()
print("Distancias ESCALADAS (edad e ingreso pesan de forma comparable):")
print(np.round(dist_escalada, 2))

### 🤔 ¿Qué acaba de pasar?

- Tomamos las columnas `Age` y `Annual_Income_k` sin transformar, y calculamos la distancia Euclidiana entre los primeros 5 clientes: como `Annual_Income_k` tiene números mucho más grandes, prácticamente decide él solo el resultado.
- `StandardScaler().fit_transform(...)` reescala ambas columnas para que tengan media 0 y una dispersión comparable.
- Con los datos escalados, las distancias cambian — ahora `Age` también influye de verdad en qué tan "cerca" o "lejos" consideramos a dos clientes.
- **Regla práctica:** casi siempre que vayas a calcular distancias (y por tanto, casi siempre antes de aplicar clustering), escala tus variables primero.

---
## 7. Resumen relámpago ⚡

| Métrica | Analogía | ¿Le importa el tamaño/escala? |
|---|---|---|
| Euclidiana | Línea recta, "como vuela el pájaro" | Sí — sensible a outliers |
| Manhattan | Un taxi moviéndose por cuadras | Sí — pero menos sensible a outliers |
| Coseno | El ángulo entre dos flechas | No — solo le importa la dirección |
| Hamming | Contar letras distintas, posición por posición | No aplica (categórica, igual longitud) |
| Estandarización (`StandardScaler`) | Convertir kilos y gramos a una misma unidad antes de comparar | — |

➡️ **Siguiente paso:** en el cuaderno [02 - K-Means Clustering (Para Dummies)](02_KMeans_Clustering_Dummies.ipynb) usarás la distancia Euclidiana (ya escalada) para que un algoritmo agrupe automáticamente a los clientes del centro comercial en segmentos con sentido de negocio.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>
